# Lesson 1: Router Engine

Welcome to Lesson 1.

To access the `requirements.txt` file, the data/pdf file required for this lesson and the `helper` and `utils` modules, please go to the `File` menu and select`Open...`.

I hope you enjoy this course!

## Setup

Run the first code cell below before anything else: it puts the repo root on `sys.path`, then loads your key via `helper.get_anthropic_api_key()` (including from `.env`).

In [22]:
import sys
from pathlib import Path

_root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "helper.py").is_file() and (p / "utils.py").is_file()
    ),
    None,
)
if _root is None:
    raise RuntimeError(
        "Could not find project root (helper.py + utils.py). "
        "Open the training repo folder or run from a lesson subfolder inside it."
    )
sys.path.insert(0, str(_root))

from helper import get_anthropic_api_key

ANTHROPIC_API_KEY = get_anthropic_api_key()

In [23]:
import nest_asyncio

nest_asyncio.apply()

## Load Data

To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

**Note**: The pdf file is included with this lesson. To access it, go to the `File` menu and select`Open...`.

In [24]:
from llama_index.core import SimpleDirectoryReader

# load documents
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()

## Define LLM and Embedding model

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

# Simply splitting based on 1024 tokens
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [26]:
from llama_index.core import Settings
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Anthropic(
    model="claude-sonnet-4-6", api_key=ANTHROPIC_API_KEY
)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

/Users/trentoncreamer/Crypto/DataCore Labs/Training/building-agentic-rag-with-llamaindex/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17581.18it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Define Summary Index and Vector Index over the Same Data

`SummaryIndex` is quick. **`VectorStoreIndex` must embed every chunk** with the HuggingFace model from the previous cell. On **CPU**, a long paper often needs **several minutes** (first run also pays one-time model download/load). The next cell prints counts and, for the vector step, a **progress bar**—that slow step is normal, not a freeze.

In [28]:
from llama_index.core import SummaryIndex, VectorStoreIndex

print(f"Chunks to index: {len(nodes)}")
print("SummaryIndex (fast)…")
summary_index = SummaryIndex(nodes)
print("VectorStoreIndex: embedding each chunk (CPU can take many minutes)…")
vector_index = VectorStoreIndex(nodes, show_progress=True)
print("Vector index ready.")

Chunks to index: 10977
SummaryIndex (fast)…
VectorStoreIndex: embedding each chunk (CPU can take many minutes)…


Generating embeddings: 100%|██████████| 737/737 [00:13<00:00, 56.35it/s]


Vector index ready.


## Define Query Engines and Set Metadata

Keep **`use_async=False`** on the summary engine in notebooks: `tree_summarize` with `use_async=True` nests asyncio inside Jupyter’s loop and typically raises **nested async** / **AsyncLibraryNotFoundError** when the router calls the summary tool.

In [29]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=False,  # True + Jupyter event loop causes nested-async errors with the router
)
vector_query_engine = vector_index.as_query_engine()

In [31]:
from llama_index.core.tools import QueryEngineTool


summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
        "Useful for summarization questions related to MetaGPT"
    ),
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
        "Useful for retrieving specific context from the MetaGPT paper."
    ),
)

## Define Router Query Engine

In [32]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector


query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True
)

In [33]:
response = query_engine.query("What is the summary of the document?")
print(str(response))

Selecting query engine 0: The question asks for a summary of the document, which aligns with choice 1 that is useful for summarization questions related to MetaGPT..


RuntimeError: Detected nested async. Please use nest_asyncio.apply() to allow nested event loops.Or, use async entry methods like `aquery()`, `aretriever`, `achat`, etc.

In [34]:
print(len(response.source_nodes))

NameError: name 'response' is not defined

In [ ]:
response = query_engine.query(
    "How do agents share information with other agents?"
)
print(str(response))

## Let's put everything together

In [ ]:
from utils import get_router_query_engine

query_engine = get_router_query_engine("metagpt.pdf")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1467.42it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Generating embeddings:  73%|███████▎  | 1500/2048 [00:29<00:10, 52.68it/s]

In [ ]:
response = query_engine.query("Tell me about the ablation study results?")
print(str(response))